In [1]:
import sys
# TO CHANGE
BASEDIR = "../../../.."
sys.path.insert(0, BASEDIR)

In [2]:
import pprint
from tqdm import tqdm

In [3]:
from src.kg_model import KnowledgeGraphModel, KnowledgeGraphModelConfig
from src.db_drivers.vector_driver import EmbedderModelConfig
from src.pipelines.memorize import MemPipelineConfig, MemPipeline, LLMUpdatorConfig
from src.pipelines.qa import QAPipeline, QAPipelineConfig

from src.pipelines.qa.query_preprocessing import QueryPreprocessorConfig
from src.pipelines.qa.answers_aggregation import AnswersAggregatorConfig
from src.pipelines.qa.kg_reasoning import KnowledgeGraphReasonerConfig

from src.db_drivers.kv_driver.configs import DEFAULT_MIXEDKV_CONFIG
from src.db_drivers.kv_driver import KeyValueDriverConfig

from src.pipelines.qa.kg_reasoning.weak_reasoner import WeakKGReasonerConfig
from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig

from src.utils import ModuleType, TripletCreator
from src.utils.data_structs import SearchPlanInfo

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Пример работы с QA-пайплайном (по построению графа знаний)

1. Инициализация модели графа знаний

In [ ]:
kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строиться не будет
)

EMBEDDER_MODEL_PATH = '../../../../models/intfloat/multilingual-e5-base' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_configs['m-e5-base'] = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)
kg_model = KnowledgeGraphModel(kg_config)

2. Инициализация Memorize-пайплайна

In [ ]:
mem_config = MemPipelineConfig(
    updator_config=LLMUpdatorConfig(
        delete_obsolete_info=False # Выключаем механизм по поиску/удалению устревших знаний в графе при добавлении новой информации
    )
)
mem_pipeline = MemPipeline(kg_model, mem_config)

In [ ]:
kg_model.clear()

3. Добавление информации в граф знаний

In [ ]:
TEXT_EXAMPLES = [
    "Sasha was walking along the highway.", 
    "Masha was walking along the highway.", 
    "The ship was sailing along the water canal.", 
    "The motorboat was sailing along the river."]

In [ ]:
extracted_triplets = []
for example in tqdm(TEXT_EXAMPLES):
    tmp_extracted_triplets, _, _ = mem_pipeline.remember(example)
    extracted_triplets += tmp_extracted_triplets

In [ ]:
kg_model.count_items()

4. Инициализация QA-пайплайна

In [ ]:
qa_config = QAPipelineConfig(
    lang='en',
    preprocessor_config=QueryPreprocessorConfig(
        denoising_config=None,
        enhancing_config=None
    ),
    reasoner_config=KnowledgeGraphReasonerConfig(
        reasoner_name='weak', reasoner_config=WeakKGReasonerConfig()
    ), 
    aggregator_config=AnswersAggregatorConfig()
)

In [ ]:
qa_config.synchronize_language()

In [ ]:
qa_pipeline = QAPipeline(kg_model, qa_config)

5. QA по графу знаний

In [ ]:
answer1, rinfo1, trace1 = qa_pipeline.answer("Did Masha walk along the highway?")
print(answer1)

In [ ]:
answer2, rinfo2, trace2 = qa_pipeline.answer("Did Katya walk along the highway?")
print(answer2)

In [ ]:
answer3, rinfo3, trace3 = qa_pipeline.answer("Did Masha and Katya walk along the highway?")
print(answer3)

In [ ]:
kg_model.clear()

In [ ]:
kg_model.close_connections()
qa_pipeline.close_connections()
del kg_model
del qa_pipeline